# Bluestock Mutual Fund Analytics — Day 2: Data Cleaning & SQL Database

This notebook demonstrates dataset cleaning across all 10 mutual fund datasets, standardizing text formatting, coercing dates, deduplicating, forward-filling missing NAV records on holidays, and loading cleaned tables into SQLite (`db/bluestock_mf.db`).

In [1]:
import sqlite3
from pathlib import Path
import pandas as pd

PROCESSED_DIR = Path('../data/processed')
DB_PATH = Path('../db/bluestock_mf.db')

print('Processed CSV Datasets Summary:')
for f in sorted(PROCESSED_DIR.glob('*.csv')):
    df = pd.read_csv(f)
    print(f'  ✓ {f.name:<35} {len(df):,} rows, {len(df.columns)} columns')

Processed CSV Datasets Summary:
  ✓ 01_fund_master.csv                  40 rows, 15 columns
  ✓ 02_nav_history.csv                  46,000 rows, 3 columns
  ✓ 03_aum_by_fund_house.csv            90 rows, 5 columns
  ✓ 04_monthly_sip_inflows.csv          48 rows, 6 columns
  ✓ 05_category_inflows.csv             144 rows, 3 columns
  ✓ 06_industry_folio_count.csv         21 rows, 6 columns
  ✓ 07_scheme_performance.csv           40 rows, 20 columns
  ✓ 08_investor_transactions.csv        32,778 rows, 13 columns
  ✓ 09_portfolio_holdings.csv           322 rows, 8 columns
  ✓ 10_benchmark_indices.csv            8,050 rows, 3 columns


## NAV History & Investor Transactions Data Integrity Checks

In [2]:
df_nav = pd.read_csv(PROCESSED_DIR / '02_nav_history.csv')
print('NAV Record Count:', len(df_nav))
print('Missing NAV Values:', df_nav['nav'].isna().sum())
print('Invalid NAV (<=0):', (df_nav['nav'] <= 0).sum())

df_tx = pd.read_csv(PROCESSED_DIR / '08_investor_transactions.csv')
print('\nTransaction Record Count:', len(df_tx))
print('Transaction Types:', df_tx['transaction_type'].value_counts().to_dict())
print('Invalid Amount (<=0):', (df_tx['amount_inr'] <= 0).sum())
print('KYC Status Split:', df_tx['kyc_status'].value_counts().to_dict())

NAV Record Count: 46000
Missing NAV Values: 0
Invalid NAV (<=0): 0

Transaction Record Count: 32778


Transaction Types: {'SIP': 19716, 'Lumpsum': 8095, 'Redemption': 4967}
Invalid Amount (<=0): 0
KYC Status Split: {'Verified': 30146, 'Pending': 2632}


## SQLite Star Schema Table Verification

In [3]:
conn = sqlite3.connect(DB_PATH)
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
print(f'Total Tables in {DB_PATH.name}:', len(tables))
for t in tables['name']:
    count = pd.read_sql(f'SELECT COUNT(*) as c FROM "{t}"', conn).iloc[0]['c']
    print(f'  ✓ {t:<30} {count:,} rows')
conn.close()

Total Tables in bluestock_mf.db: 21
  ✓ aum_by_fund_house              90 rows
  ✓ benchmark_indices              8,050 rows
  ✓ category_inflows               144 rows
  ✓ dim_date                       1,296 rows
  ✓ dim_fund                       40 rows
  ✓ fact_aum                       90 rows
  ✓ fact_benchmark                 8,050 rows
  ✓ fact_category_inflows          144 rows
  ✓ fact_industry_folio            21 rows
  ✓ fact_nav                       46,000 rows
  ✓ fact_performance               40 rows
  ✓ fact_portfolio                 322 rows
  ✓ fact_sip_industry              48 rows
  ✓ fact_transactions              32,778 rows
  ✓ fund_master                    40 rows
  ✓ industry_folio_count           21 rows
  ✓ investor_transactions          32,778 rows
  ✓ monthly_sip_inflows            48 rows
  ✓ nav_history                    46,000 rows
  ✓ portfolio_holdings             322 rows
  ✓ scheme_performance             40 rows
